# Set WD to Root

In [2]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

# print(Path.cwd())

# Load Hyperparameters

In [30]:
# Load hyperparamters
import json

PARAM_PATH = Path('./config/test_params.json')

with open(PARAM_PATH, "r", encoding="utf-8") as f:
    params = json.load(f)

print(params)

{'batch_size': 200, 'learning_rate': 0.01, 'epochs': 5000, 'hidden_size': 100}


# Load Data Loader

In [31]:
from scripts.data_loader.piqa_dataloader import get_piqa_dataloaders
from pathlib import Path
import sentencepiece as spm

BASE_PATH = Path('./datasets/PIQA')
BASE_PATH = BASE_PATH.resolve()

tokenizer = spm.SentencePieceProcessor(
    model_file="./scripts/tokenizer/piqa_bpe.model"
)

train_loader, validate_loader, test_loader = get_piqa_dataloaders(base_path=BASE_PATH,
                                                                  tokenizer=tokenizer,
                                                                  batch_size=200)

In [32]:
for goal_ids, goal_mask, sol_ids, sol_mask, labels in train_loader:
    print(goal_ids.shape, sol_ids.shape)
    break

torch.Size([200, 22]) torch.Size([200, 2, 192])


In [42]:
import torch

batch_no = 0

for goal_ids, goal_mask, sol_ids, sol_mask, labels in train_loader:

    lengths = sol_mask.sum(dim=-1)
    indices = torch.where(lengths > 400)

    for i, j in zip(*indices):
        i = i.item()
        j = j.item()

        tokens = sol_ids[i, j][sol_mask[i, j]].tolist()

        print(f"\nBatch {batch_no}, sample {i}, candidate {j}")
        print("length:", len(tokens))
        print(tokenizer.decode_ids(tokens))

    batch_no += 1
  


Batch 59, sample 46, candidate 0
length: 470
To make basic meatballs, you’ll need 1/2 cup milk, 1/2 cup fine breadcrumbs, 1 large egg, 1 teaspoon salt, freshly ground black pepper, 1/2 cup grated Parmesan cheese, 1/4 cup finely minced Italian parsley, 1 pound ground meat (such as beef, pork, turkey, chicken, or veal, or a mix), 1/2 cup finely chopped onion (or grated on a coarse grater) from 1 small yellow onion, and 1 clove garlic Combine the milk and breadcrumbs. Place the breadcrumbs in a small bowl, pour in the milk, and stir to combine. Set aside while preparing the rest of the meatball mixture. The breadcrumbs will absorb the milk and become soggy. Whisk the egg, salt, pepper, Parmesan, and parsley. Whisk the egg in a large bowl until broken up. Add the Parmesan, parsley, salt, and a generous quantity of black pepper, and whisk to combine. Add the ground meat. Add the meat to the egg mixture. Use your hands to thoroughly mix the egg mixture into the ground meat. Add the onions a

In [39]:
import torch

lengths = sol_mask.sum(dim=-1)

max_length = lengths.max()

indices = torch.where(lengths == max_length)

print("max length:", max_length.item())

for i, j in zip(indices[0], indices[1]):
    print(
        "sample:", i.item(),
        "candidate:", j.item(),
        "length:", lengths[i, j].item()
    )

max length: 192
sample: 92 candidate: 0 length: 192


# Feed Data to Model

In [34]:
from scripts.models.bigru_attention import BiGRUAttention
import torch.optim as optim
import torch.nn.functional as F

bigru_model = BiGRUAttention(vocab_size=tokenizer.vocab_size(),
                       pad_id=tokenizer.pad_id(), use_goal_attn=True)

bigru_optimiser = optim.SGD(params=bigru_model.parameters(), lr=params['learning_rate'])
criterion = F.cross_entropy

In [18]:
from time import perf_counter

bigru_model.train()

epoch_loss = 0.0

for goal_ids, goal_mask, sol_ids, sol_mask, labels in train_loader:

    start_ts = perf_counter()
    # Forward
    bigru_optimiser.zero_grad()

    logits, _ = bigru_model.forward(goal_ids, goal_mask, sol_ids, sol_mask, return_attn=False)

    # Loss
    loss = criterion(
        input=logits,
        target=labels
    )

    # Backward
    loss.backward()

    bigru_optimiser.step()

    # Record loss
    epoch_loss += loss.item()

    end_ts = perf_counter()
    time_elips = end_ts - start_ts
    
    print(
        f"B={sol_ids.shape[0]}, "
        f"Lg={goal_ids.shape[1]}, "
        f"Ls={sol_ids.shape[2]}, "
        f"time={time_elips:.2f}s"
    )


B=200, Lg=26, Ls=143, time=13.66s
B=200, Lg=24, Ls=90, time=7.41s
B=200, Lg=31, Ls=150, time=11.95s
B=200, Lg=23, Ls=187, time=15.70s
B=200, Lg=25, Ls=110, time=8.97s


KeyboardInterrupt: 